# Prerequisites

In [0]:
from pyspark.sql.functions import *
# customers_df = spark.read.table("ecommerce.silver.customers")
orders_df = spark.read.table("ecommerce.silver.orders")
order_items_df = spark.read.table("ecommerce.silver.order_items")
products_df = spark.read.table("ecommerce.silver.products")
# payments_df = spark.read.table("ecommerce.silver.payments")
# inventory_df = spark.read.table("ecommerce.silver.inventory")
# clickstream_df = spark.read.table("ecommerce.silver.clickstream")

# Calculations

## join product , orders and order_items table

In [0]:
products_perf = products_df.join(order_items_df, "product_id").join(
    orders_df, "order_id"
)
# products_perf.printSchema()
products_perf.limit(5).display()

## units_sold, total_orders, gross_revenue, discount_amount, net_revenue, unique_customers, average_selling_price

In [0]:
products_perf = products_perf.groupBy("product_id", "product_name", "category").agg(
    sum(when(col("order_status") == lit("DELIVERED"), col("quantity"))).alias(
        "units_sold"
    ),
    countDistinct("order_id").alias("total_orders"),
    round(sum(col("gross_amount")),2).alias("gross_revenue"),
    round(sum(col("net_amount")),2).alias("net_revenue"),
    round(sum(col("discount_amount")),2).alias("discount_amount"),
    countDistinct("customer_id").alias("unique_customers"),
).withColumn("avg_selling_price", round(col("gross_revenue")/col("units_sold"),2))

products_perf.limit(5).display()

## revenue_rank, category_rank, using Spark window functions

In [0]:
from pyspark.sql.window import Window

window_category = Window.partitionBy("category").orderBy(desc("net_revenue"))
window_revenue = Window.orderBy(desc("net_revenue"))
products_perf = products_perf.withColumn("category_rank", dense_rank().over(window_category)).withColumn("revenue_rank", dense_rank().over(window_revenue))

products_perf.limit(5).display()


# Create gold.products_perf table

In [0]:
products_perf.write.mode("overwrite").option("mergeSchema","true").saveAsTable("ecommerce.gold.product_performance")
spark.read.table("ecommerce.gold.product_performance").limit(5).display()